# Converting M2HATS high-rate netCDF to Zarr

This notebook walks through using `m2hats_to_zarr.py` to convert the flat ISFS netCDF naming (`<var>_<height>_<site>`) into a CF-compliant Zarr DataTree with real `site` / `height` dimensions, sub-second sample offsets, and validity flags.

The notebook also handles the **t1 CSAT instrument swap on 2023-08-08 13:00 UTC** (CSAT3 at 30 Hz → CSAT3A at 60 Hz) by splitting the conversion at the boundary and unifying the two halves at write time, so the resulting Zarr is a single coherent store with no visible seam.

**Before you run this**, make sure `m2hats_to_zarr.py` is sitting next to this notebook (or somewhere on your Python path).

**Dependencies**: `xarray>=2024.10` (for `DataTree`), `netCDF4`, `zarr`, `numcodecs`, `dask`, `dask-jobqueue` (if you're on a PBS cluster like NCAR Casper).

In [ ]:
import sys, pathlib, glob
sys.path.insert(0, str(pathlib.Path.cwd()))
from pathlib import Path

import numpy as np
import xarray as xr
import m2hats_to_zarr as m
from m2hats_to_zarr import restructure_m2hats

ModuleNotFoundError: No module named 'm2hats_to_zarr'

## 1. Connect a Dask cluster

Creating a `PBSCluster` object only describes what a worker job looks like — it does **not** start workers and does **not** wire dask into your kernel. You need a `Client` and a `scale()` call before anything will actually run distributed.

Skip this section if you're working with a small subset and a local-threaded scheduler is fine. For the full ~250 GB conversion, distributed execution is essential.

In [ ]:
from dask_jobqueue import PBSCluster
from dask.distributed import Client

lustre_scratch  = "/lustre/desc1/scratch/myasears"

cluster = PBSCluster(
    job_name = 'dask-eol-26',
    cores = 1,
    memory = '8GiB',
    processes = 1,
    local_directory = lustre_scratch+'/dask/spill/',
    log_directory = lustre_scratch + '/dask/logs/',
    resource_spec = 'select=1:ncpus=1:mem=8GB',
    queue = 'casper',
    walltime = '5:00:00',
    interface = 'ext'
)

client = Client(cluster)
cluster.scale(jobs=20)
client.wait_for_workers(20)
print(client)

## 2. Fix errors in the data

In [ ]:
isfs_highrate_path = "/lustre/desc1/scratch/myasears/M2HATS/FDA_datasets/surface_highrate_geo"

In [ ]:
normal_path = f"{isfs_highrate_path}/isfs_m2hats_qc_geo_hr_20230728_03.nc"
normal_dataset = xr.open_dataset(normal_path)
print(normal_dataset.dims)

errored_path = f"{isfs_highrate_path}/isfs_m2hats_qc_geo_hr_20230728_06.nc"
errored_dataset = xr.open_dataset(errored_path)
print(errored_dataset.dims)

In [ ]:
# Rename dims so they match the others:
#   sample (50)    -> sample_50
#   sample_60 (60) -> sample
# Use a temporary name to avoid a name collision during the swap.
errored_dataset = errored_dataset.rename({"sample": "__tmp_sample_50"})
errored_dataset = errored_dataset.rename({"sample_60": "sample"})
errored_dataset = errored_dataset.rename({"__tmp_sample_50": "sample_50"})

errored_dataset.to_netcdf(f"{isfs_highrate_path}/isfs_m2hats_qc_geo_hr_20230728_06_aligned.nc")

In [ ]:
# Locate and select files to be read in, replacing the errored file with the aligned version.
isfs_highrate_files = [
    f for f in sorted(Path(isfs_highrate_path).glob("*.nc"))
    if f.name != "isfs_m2hats_qc_geo_hr_20230728_06.nc"
]

## 2. Open the source netCDFs

Two things to know about the M2HATS hourly files:

1. Each file's `time` coord is anchored at hour-0-of-the-day (i.e. `2023-MM-DDT00:00:00.5` to `T00:59:59.5`) regardless of which hour the file actually represents. The actual hour lives in the scalar `base_time`. Default xarray decoding gets this wrong — every file looks like the 0th hour of the day. The `fix_time` preprocess below corrects it.

2. The filenames sort alphabetically into chronological order (`YYYYMMDD_HH`), so we use `combine="nested"` with an explicit sort. This avoids `combine="by_coords"`'s slow post-open analysis pass — on 1500 files it shaves minutes off the open.

In [ ]:
def fix_time(ds):
    """Override `time` with base_time + within-hour offset.

    M2HATS high-rate files store `time` as 'seconds within hour 0 of the
    file's day' rather than absolute, and `base_time` holds the hour offset.
    Defensive: if base_time is missing, or if `time` already exceeds the
    hour-0 range (meaning the file is already correctly decoded), pass
    through unchanged.
    """
    if "base_time" not in ds.variables:
        return ds
    if (ds["time"].dt.hour > 0).any():
        return ds.drop_vars("base_time", errors="ignore")
    base = ds["base_time"]
    tod = ds["time"] - ds["time"].dt.floor("D")
    return ds.assign_coords(time=base + tod).drop_vars("base_time")

In [ ]:
# Point this at wherever your hourly files live.
SRC_GLOB = "/path/to/isfs_m2hats_qc_geo_tiltcor_hr_*.nc"
ZARR_OUT = "./m2hats.zarr"

isfs_highrate_files = sorted(glob.glob(SRC_GLOB))   # alphabetic == chronological
print(f"{len(isfs_highrate_files)} files")

ds = xr.open_mfdataset(
    isfs_highrate_files,
    preprocess=fix_time,
    combine="nested",                # skip by_coords' slow ordering pass
    concat_dim="time",
    parallel=True,
    chunks={"time": 3600},           # one file = one chunk along time
    decode_timedelta=False,
    data_vars="minimal",
    coords="minimal",
    compat="override",               # skip per-pair schema checks
)
print(f"{len(ds.data_vars):,} variables, dims = {dict(ds.sizes)}")

### Quick sanity check on the combined time axis

If you trust filename-sort instead of coord-based sort, verify the result is monotonic and the right length before moving on. Gaps larger than 1 second mean missing hourly files.

In [ ]:
diff = ds.time.diff("time")
print(f"first/last time:    {ds.time.values[0]}  /  {ds.time.values[-1]}")
print(f"monotonic:          {bool((diff > np.timedelta64(0)).all())}")
print(f"smallest step:      {diff.min().values}")
print(f"largest step:       {diff.max().values}")
print(f"expected n times:   {3600 * len(isfs_highrate_files):,}")
print(f"actual n times:     {ds.sizes['time']:,}")

## 3. Smoke-test the restructure on a one-hour slice

Strongly recommended before any full conversion. Two reasons for `.persist()` here: it pulls the one hour into distributed memory (so the slice's dask graph stops referencing all 1500 source files), and it keeps subsequent graph-build operations in the restructure fast.

If anything's off — variable naming surprise, validity flag misaligned with UTC, dim signature wrong — you'd rather find out in seconds than 3 hours into a 250 GB write.

In [ ]:
subset = ds.isel(time=slice(0, 3600)).persist()    # one hour, in cluster memory

tree_test = restructure_m2hats(
    subset,
    output_path="./m2hats_test.zarr",
    tilt_corrected=True,
    time_chunk_seconds=3600,
    write=False,                  # in-memory only for now
)

for path in tree_test.groups:
    node = tree_test[path].ds
    if not node.data_vars:
        continue
    print(f"{path:35s} dims={dict(node.sizes)}")

Expected output on a pre-Aug-8 hour (or any subset that doesn't span the t1 swap):

```
/array/sonic_60hz           dims={'site': 21, 'time': 3600, 'sample': 60}
/array/sonic_50hz           dims={'site': 9,  'time': 3600, 'sample_50': 50}
/array/sonic_30hz           dims={'site': 20, 'time': 3600, 'sample_30': 30}
/array/irga_60hz            dims={'site': 16, 'time': 3600, 'sample': 60}
/array/barometer_20hz       dims={'site': 16, 'time': 3600, 'sample_20': 20}
/array/trh_1hz              dims={'site': 16, 'time': 3600}
/profile_t0/sonic_60hz      dims={'height': 8, 'time': 3600, 'sample': 60}
/profile_t0/irga_60hz       dims={'height': 8, 'time': 3600, 'sample': 60}
/profile_t0/barometer_20hz  dims={'height': 8, 'time': 3600, 'sample_20': 20}
/profile_t0/trh_1hz         dims={'height': 8, 'time': 3600}
```

21 + 9 + 20 = 50 array towers, always. If you see an unexpected extra `sample` dim leaking into `sonic_30hz` or `sonic_50hz` it means your subset spans the t1 swap boundary — the next section handles that.

Spot-check a few real values to be sure the data isn't all NaN:

In [ ]:
leaf = tree_test["/array/sonic_60hz"].ds
print(leaf.wind_u.isel(site=0, time=slice(0, 5)).compute())

## 4. Full conversion: split at the t1 swap boundary

The t1 sonic was a CSAT3 (30 Hz) until 2023-08-08 13:00 UTC and a CSAT3A (60 Hz) after. A naive single-pass restructure produces dual-sample-dim arrays where xarray concat had to broadcast across the swap. To avoid that, we split `ds` at the boundary, restructure each half independently, then **unify the site coords and append along time** when writing. Downstream users see a single coherent Zarr — t1's pre-swap data lives in `sonic_30hz` and post-swap data in `sonic_60hz`, with NaN padding in the off-period of each.

In [ ]:
SWAP = np.datetime64("2023-08-08T13:00:00")

# Lazy split — neither half is computed yet.
pre  = ds.sel(time=slice(None, SWAP - np.timedelta64(1, "ns")))
post = ds.sel(time=slice(SWAP, None))

print(f"pre:  {pre.sizes['time']:>8,} time steps  ({pre.time.values[0]}  ..  {pre.time.values[-1]})")
print(f"post: {post.sizes['time']:>8,} time steps  ({post.time.values[0]}  ..  {post.time.values[-1]})")

# Restructure each half (still lazy).
tree_pre  = restructure_m2hats(pre,  ZARR_OUT, write=False, tilt_corrected=True)
tree_post = restructure_m2hats(post, ZARR_OUT, write=False, tilt_corrected=True)

In [ ]:
def unify_axes(pre_ds, post_ds):
    """Reindex pre and post to the union of site (or height) coords, then
    refill auxiliary site/height coords from the script's static tables so
    the NaN-padded entries have correct metadata."""
    if "site" in pre_ds.dims or "site" in post_ds.dims:
        expand = "site"
    else:
        expand = "height"

    pre_keys  = set(pre_ds[expand].values.tolist())  if expand in pre_ds.dims  else set()
    post_keys = set(post_ds[expand].values.tolist()) if expand in post_ds.dims else set()

    if expand == "site":
        keys = sorted(pre_keys | post_keys, key=m._site_sort_key)
        aux = dict(
            site_height=("site", [m.ARRAY_SITES[s][0] for s in keys]),
            site_lon   =("site", [m.ARRAY_SITES[s][1] for s in keys]),
            site_lat   =("site", [m.ARRAY_SITES[s][2] for s in keys]),
            csat_model_appendix_a=("site",
                [m.CSAT_MODEL.get(s, "unknown") for s in keys]),
        )
    else:
        keys = sorted(pre_keys | post_keys)
        aux = dict(
            height_actual=("height", [m.T0_PROFILE[h][0] for h in keys]),
            height_lon   =("height", [m.T0_PROFILE[h][1] for h in keys]),
            height_lat   =("height", [m.T0_PROFILE[h][2] for h in keys]),
        )

    return (pre_ds.reindex({expand: keys}).assign_coords(**aux),
            post_ds.reindex({expand: keys}).assign_coords(**aux))

In [ ]:
# Stamp the swap into the root attrs so the store is self-documenting.
tree_pre.attrs["t1_csat_swap_utc"] = "2023-08-08T13:00:00"
tree_pre.attrs["t1_csat_swap_note"] = (
    "Site t1 sonic was a CSAT3 (30 Hz) before this timestamp and a CSAT3A "
    "(60 Hz) after. Pre-swap data lives in /array/sonic_30hz; post-swap "
    "data lives in /array/sonic_60hz. Each group's t1 column is NaN-padded "
    "outside its valid period."
)

# Group-by-group write: pre first, post appended along time.
# This keeps the dask graph small per group and gives clear progress.
mode = "w"
for path, leaf_pre in tree_pre.subtree_with_keys:
    if not leaf_pre.ds.data_vars:
        continue
    leaf_post_ds = tree_post[path].ds if path in tree_post else None

    if leaf_post_ds is None or not leaf_post_ds.data_vars:
        # Group only exists in pre — straight write.
        print(f"writing  {path:30s} ({leaf_pre.ds.sizes['time']:>9,} steps)", flush=True)
        leaf_pre.ds.to_zarr(ZARR_OUT, group=path, mode=mode, consolidated=False)
    else:
        pre_a, post_a = unify_axes(leaf_pre.ds, leaf_post_ds)
        print(f"writing  {path:30s} pre  ({pre_a.sizes['time']:>9,} steps)", flush=True)
        pre_a.to_zarr(ZARR_OUT, group=path, mode=mode, consolidated=False)
        print(f"append   {path:30s} post ({post_a.sizes['time']:>9,} steps)", flush=True)
        post_a.to_zarr(ZARR_OUT, group=path, mode="a",
                       append_dim="time", consolidated=False)
    mode = "a"

import zarr
zarr.consolidate_metadata(ZARR_OUT)
print("done.")

## 5. Re-open the Zarr store

This is what every downstream user will do. No need to run the conversion script again, and no need to know about the t1 swap — it's already absorbed into the store.

In [ ]:
tree = xr.open_datatree(ZARR_OUT, engine="zarr")
print(tree)
print("\nt1 swap note:", tree.ds.attrs.get("t1_csat_swap_note", "(not present)"))

## 6. The payoff: queries that used to require string parsing

In [ ]:
arr = tree["/array/sonic_60hz"].ds

# (a) Pull one site, no string surgery required.
t17 = arr.sel(site="t17")
print(t17.wind_u)

# (b) Cross-array statistic: mean U across all towers, the whole time series.
u_array_mean = arr.wind_u.mean(dim="site")
print("\nArray-averaged wind_u shape:", u_array_mean.shape)

# (c) Just the towers that also carry an IRGA. Detect by asking which sites
#     appear in both sonic_60hz and irga_60hz.
irga_sites = set(tree["/array/irga_60hz"].ds.site.values.tolist())
sonic_irga = arr.sel(site=[s for s in arr.site.values if s in irga_sites])
print("\nSonics at IRGA-equipped sites:", list(sonic_irga.site.values))

In [ ]:
# (d) t1 across the swap. Pre-swap data lives in sonic_30hz; post-swap in
#     sonic_60hz. Concat them along time after collapsing the sample dim
#     to get a continuous 1-Hz wind_u record at t1.
t1_pre  = tree["/array/sonic_30hz"].ds.sel(site="t1").wind_u.mean("sample_30")
t1_post = tree["/array/sonic_60hz"].ds.sel(site="t1").wind_u.mean("sample")
t1_continuous = xr.concat([t1_pre.dropna("time"),
                            t1_post.dropna("time")], dim="time")
print("t1 continuous record (1 Hz):", t1_continuous.sizes)

In [ ]:
# (e) Vertical wind profile from the t0 multi-level tower.
prof = tree["/profile_t0/sonic_60hz"].ds

# 30-minute mean wind speed at each height, only when the t0 group is valid
# (skips the pre-31-July move period and all five tower-lowerings).
spd = np.sqrt(prof.wind_u**2 + prof.wind_v**2 + prof.wind_w**2)
spd_30min = (
    spd.where(prof.valid == 0)
       .mean("sample")              # collapse 60 Hz -> 1 per second
       .resample(time="30min").mean()
)
print(spd_30min)

In [ ]:
# (f) Reconstructing the true high-rate timestamp.
# `time` is the 1-Hz bin center; `sample_offset` is the centered sub-second
# offset of each high-rate sample. Adding them gives the actual obs time.
high_rate_time = arr.time + (arr.sample_offset * 1e9).astype("timedelta64[ns]")
print(high_rate_time.shape, "  e.g. first second's 60 samples:")
print(high_rate_time.isel(time=0).values[:3], "...",
      high_rate_time.isel(time=0).values[-3:])

In [ ]:
# (g) Flux-style covariance (eddy-covariance w'T' at each site, per 30 min).
# Detrend by subtracting the per-window mean across both time and sample,
# then average the product. This is the standard sensible-heat-flux kernel
# (still needs Schotanus correction for humidity to be true SH).
block = arr.resample(time="30min")
w_anom  = arr.wind_w           - block.mean()["wind_w"]
tc_anom = arr.sonic_temperature - block.mean()["sonic_temperature"]
w_tc = (w_anom * tc_anom).resample(time="30min").mean(dim=["sample", "time"])
print(w_tc)

## 7. Sanity checks worth running on every conversion

In [ ]:
# Any source vars the script couldn't route?
print("skipped:", tree.ds.attrs.get("skipped_source_vars") or "(none)")

# Does the validity flag fire as expected during the Sep-3 battery swap?
prof = tree["/profile_t0/sonic_60hz"].ds
during_swap = prof.sel(time=slice("2023-09-03T14:30", "2023-09-03T17:30"))
print("\nvalid-flag values during Sep-3 swap window:",
      sorted(set(during_swap.valid.values.tolist())))
print("(expect 0 and 3 — 3 = battery_swap)")

# t1 NaN pattern: pre-swap data should be present in sonic_30hz and absent
# in sonic_60hz; post-swap is the opposite.
t1_30 = tree["/array/sonic_30hz"].ds.sel(site="t1").wind_u
t1_60 = tree["/array/sonic_60hz"].ds.sel(site="t1").wind_u
print("\nt1 in sonic_30hz: "
      f"finite before swap = {int(t1_30.sel(time=slice(None, '2023-08-08T13:00')).notnull().sum())}, "
      f"finite after = {int(t1_30.sel(time=slice('2023-08-08T13:00', None)).notnull().sum())}")
print("t1 in sonic_60hz: "
      f"finite before swap = {int(t1_60.sel(time=slice(None, '2023-08-08T13:00')).notnull().sum())}, "
      f"finite after = {int(t1_60.sel(time=slice('2023-08-08T13:00', None)).notnull().sum())}")
print("(expect: 30hz has data only before; 60hz has data only after)")